In [2]:
import os
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import chisquare, norm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
SQL_DIR = PROJECT_ROOT / "sql"
WRITEUP_DIR = PROJECT_ROOT / "docs" / "writeups"
os.chdir(PROJECT_ROOT)

con = duckdb.connect(str(DATA_DIR / "criteo.duckdb"))
print(f"Connected. Row count: {con.execute('SELECT COUNT(*) FROM criteo').fetchone()[0]:,}")

Connected. Row count: 13,979,592


In [2]:
n_t, n_c = con.execute("""
    SELECT
        SUM(CAST(treatment AS BIGINT))     AS n_treated,
        SUM(1 - CAST(treatment AS BIGINT)) AS n_control
    FROM criteo
""").fetchone()
n_total = n_t + n_c
print(f"Treated: {n_t:,}")
print(f"Control: {n_c:,}")
print(f"Total:   {n_total:,}")
print(f"Observed treated share: {n_t / n_total:.6f}")

Treated: 11,882,655
Control: 2,096,937
Total:   13,979,592
Observed treated share: 0.850000


In [3]:
# Per Criteo documentation, target split is 85% treated / 15% control
EXPECTED_TREATED_SHARE = 0.85
exp_t = n_total * EXPECTED_TREATED_SHARE
exp_c = n_total * (1 - EXPECTED_TREATED_SHARE)

chi2, p_value = chisquare(f_obs=[n_t, n_c], f_exp=[exp_t, exp_c])

print(f"Expected treated: {exp_t:,.0f}")
print(f"Expected control: {exp_c:,.0f}")
print(f"Observed treated: {n_t:,}")
print(f"Observed control: {n_c:,}")
print()
print(f"Chi-square statistic: {chi2:,.2f}")
print(f"p-value:              {p_value:.4g}")
print()
if p_value < 0.001:
    print("⚠️  SRM DETECTED (p < 0.001). Randomization is suspect.")
else:
    print("✅ No SRM. Observed split is consistent with the documented 85/15.")

Expected treated: 11,882,653
Expected control: 2,096,939
Observed treated: 11,882,655
Observed control: 2,096,937

Chi-square statistic: 0.00
p-value:              0.9989

✅ No SRM. Observed split is consistent with the documented 85/15.


In [4]:
# Cohen's w for proportion deviation: a more interpretable companion to p-value
observed_share = n_t / n_total
deviation_pp = (observed_share - EXPECTED_TREATED_SHARE) * 100
relative_deviation = (observed_share - EXPECTED_TREATED_SHARE) / EXPECTED_TREATED_SHARE

print(f"Observed treated share: {observed_share:.6f}")
print(f"Expected treated share: {EXPECTED_TREATED_SHARE:.6f}")
print(f"Absolute deviation:     {deviation_pp:+.4f} percentage points")
print(f"Relative deviation:     {relative_deviation:+.4%}")

Observed treated share: 0.850000
Expected treated share: 0.850000
Absolute deviation:     +0.0000 percentage points
Relative deviation:     +0.0000%


In [5]:
ate_sql = """-- ATE for visit and conversion outcomes.
-- Computed as difference of two proportions with analytic SE.
-- Run from project root: duckdb data/criteo.duckdb < sql/03_ate.sql

WITH outcome_stats AS (
    SELECT
        outcome,
        SUM(CASE WHEN treatment = 1 THEN y ELSE 0 END) * 1.0
            / SUM(CASE WHEN treatment = 1 THEN 1 ELSE 0 END) AS p_t,
        SUM(CASE WHEN treatment = 0 THEN y ELSE 0 END) * 1.0
            / SUM(CASE WHEN treatment = 0 THEN 1 ELSE 0 END) AS p_c,
        SUM(CASE WHEN treatment = 1 THEN 1 ELSE 0 END)        AS n_t,
        SUM(CASE WHEN treatment = 0 THEN 1 ELSE 0 END)        AS n_c
    FROM (
        SELECT 'visit' AS outcome,      treatment, visit       AS y FROM criteo
        UNION ALL
        SELECT 'conversion' AS outcome, treatment, conversion  AS y FROM criteo
    ) t
    GROUP BY outcome
)
SELECT
    outcome,
    p_c                                                AS control_rate,
    p_t                                                AS treatment_rate,
    p_t - p_c                                          AS ate,
    SQRT( p_t * (1 - p_t) / n_t
        + p_c * (1 - p_c) / n_c )                      AS se,
    (p_t - p_c) - 1.96 * SQRT( p_t * (1 - p_t) / n_t
                              + p_c * (1 - p_c) / n_c) AS ci_low,
    (p_t - p_c) + 1.96 * SQRT( p_t * (1 - p_t) / n_t
                              + p_c * (1 - p_c) / n_c) AS ci_high,
    n_t,
    n_c
FROM outcome_stats
ORDER BY outcome;
"""

ate_path = SQL_DIR / "03_ate.sql"
ate_path.write_text(ate_sql)
print(f"Wrote {ate_path} ({len(ate_sql):,} chars)")

Wrote /Users/madhumithakatam/Documents/Projects/CriteoUpliftModeling/criteo_uplift_modeling/sql/03_ate.sql (1,462 chars)


In [6]:
%%time
ate_df = con.execute(ate_sql).fetchdf()
ate_df

CPU times: user 285 ms, sys: 25.2 ms, total: 310 ms
Wall time: 58 ms


,outcome,control_rate,treatment_rate,ate,se,ci_low,ci_high,n_t,n_c
0,conversion,0.001938,0.003089,0.001152,0.000034,0.001085,0.001219,11882655.0,2096937.0
1,visit,0.038201,0.048543,0.010342,0.000146,0.010056,0.010629,11882655.0,2096937.0


In [7]:
ate_df["z"]       = ate_df["ate"] / ate_df["se"]
ate_df["p_value"] = 2 * (1 - norm.cdf(np.abs(ate_df["z"])))

# Reorder for the writeup table
display_cols = ["outcome", "control_rate", "treatment_rate", "ate",
                "se", "ci_low", "ci_high", "z", "p_value", "n_t", "n_c"]
ate_df = ate_df[display_cols]

# Pretty-print
pd.set_option("display.float_format", lambda x: f"{x:.6f}")
ate_df

,outcome,control_rate,treatment_rate,ate,se,ci_low,ci_high,z,p_value,n_t,n_c
0,conversion,0.001938,0.003089,0.001152,0.000034,0.001085,0.001219,33.512267,0.000000,11882655.000000,2096937.000000
1,visit,0.038201,0.048543,0.010342,0.000146,0.010056,0.010629,70.685184,0.000000,11882655.000000,2096937.000000


In [8]:
ate_df["relative_lift"] = ate_df["ate"] / ate_df["control_rate"]
ate_df[["outcome", "control_rate", "treatment_rate", "ate",
        "ci_low", "ci_high", "p_value", "relative_lift"]]

,outcome,control_rate,treatment_rate,ate,ci_low,ci_high,p_value,relative_lift
0,conversion,0.001938,0.003089,0.001152,0.001085,0.001219,0.000000,0.594488
1,visit,0.038201,0.048543,0.010342,0.010056,0.010629,0.000000,0.270737


In [10]:
!pip install tabulate

  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
Using cached tabulate-0.10.0-py3-none-any.whl (39 kB)


In [11]:
results_path = WRITEUP_DIR / "ate_results.csv"
ate_df.to_csv(results_path, index=False)
print(f"Saved {results_path}")

# Build a markdown table for direct paste into the writeup
md_table = ate_df[["outcome", "control_rate", "treatment_rate", "ate",
                   "ci_low", "ci_high", "p_value"]].copy()
for col in ["control_rate", "treatment_rate", "ate", "ci_low", "ci_high"]:
    md_table[col] = md_table[col].apply(lambda x: f"{x:.6f}")
md_table["p_value"] = md_table["p_value"].apply(lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}")
print(md_table.to_markdown(index=False))

Saved /Users/madhumithakatam/Documents/Projects/CriteoUpliftModeling/criteo_uplift_modeling/docs/writeups/ate_results.csv
| outcome    |   control_rate |   treatment_rate |      ate |   ci_low |   ci_high |   p_value |
|:-----------|---------------:|-----------------:|---------:|---------:|----------:|----------:|
| conversion |       0.001938 |         0.003089 | 0.001152 | 0.001085 |  0.001219 |         0 |
| visit      |       0.038201 |         0.048543 | 0.010342 | 0.010056 |  0.010629 |         0 |


In [12]:
con.close()
print("Connection closed.")

Connection closed.


In [3]:
# Skip if connection is still open from Day 4
con = duckdb.connect(str(DATA_DIR / "criteo.duckdb"))
print(f"Connected. Row count: {con.execute('SELECT COUNT(*) FROM criteo').fetchone()[0]:,}")

Connected. Row count: 13,979,592


In [4]:
features = [f"f{i}" for i in range(12)]

def heterogeneity_block(feat: str) -> str:
    """CTE-based bucketing and lift calc for one feature."""
    return f"""SELECT
    '{feat}' AS feature_name,
    bucket,
    treatment,
    AVG(conversion)        AS rate,
    COUNT(*)               AS n,
    SUM(conversion)        AS conversions
FROM (
    SELECT
        conversion,
        treatment,
        NTILE(4) OVER (ORDER BY {feat}) AS bucket
    FROM criteo
)
GROUP BY bucket, treatment"""

header = """-- Auto-generated by notebooks/02_experimentation.ipynb (Day 5).
-- Quartile-bucketed lift per feature for heterogeneity analysis.
-- One row per (feature, bucket, treatment). Pivot in pandas to get
-- per-bucket lift = treatment_rate - control_rate.
-- Run from project root: duckdb data/criteo.duckdb < sql/04_heterogeneity.sql

"""

het_sql = header + "\nUNION ALL\n".join(heterogeneity_block(f) for f in features) + "\nORDER BY feature_name, bucket, treatment;\n"

het_path = SQL_DIR / "04_heterogeneity.sql"
het_path.write_text(het_sql)
print(f"Wrote {het_path} ({len(het_sql):,} chars, {len(features)} feature blocks)")

Wrote /Users/madhumithakatam/Documents/Projects/CriteoUpliftModeling/criteo_uplift_modeling/sql/04_heterogeneity.sql (4,315 chars, 12 feature blocks)


In [5]:
%%time
het_long = con.execute(het_sql).fetchdf()
print(f"Rows: {len(het_long)}  (expected: 12 features × 4 buckets × 2 arms = 96)")
het_long.head(8)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 96  (expected: 12 features × 4 buckets × 2 arms = 96)
CPU times: user 15.6 s, sys: 2.3 s, total: 17.9 s
Wall time: 2.9 s


,feature_name,bucket,treatment,rate,n,conversions
0,f0,1,0,0.001490,557576,831.0
1,f0,1,1,0.001957,2937322,5749.0
2,f0,2,0,0.005651,482557,2727.0
3,f0,2,1,0.009046,3012341,27250.0
4,f0,3,0,0.000609,526737,321.0
5,f0,3,1,0.000803,2968161,2384.0
6,f0,4,0,0.000347,530067,184.0
7,f0,4,1,0.000448,2964831,1328.0


In [6]:
# Pivot rate
rate_wide = het_long.pivot_table(
    index=["feature_name", "bucket"],
    columns="treatment",
    values="rate",
).rename(columns={0: "control_rate", 1: "treatment_rate"})

# Pivot n
n_wide = het_long.pivot_table(
    index=["feature_name", "bucket"],
    columns="treatment",
    values="n",
).rename(columns={0: "n_c", 1: "n_t"}).astype(int)

# Combine
het_wide = rate_wide.join(n_wide).reset_index()
het_wide.columns.name = None
het_wide["lift"] = het_wide["treatment_rate"] - het_wide["control_rate"]

# Standard error of difference of two proportions, per bucket
het_wide["lift_se"] = np.sqrt(
    het_wide["treatment_rate"] * (1 - het_wide["treatment_rate"]) / het_wide["n_t"]
    + het_wide["control_rate"] * (1 - het_wide["control_rate"]) / het_wide["n_c"]
)

het_wide = het_wide[["feature_name", "bucket", "control_rate", "treatment_rate",
                     "lift", "lift_se", "n_t", "n_c"]]
het_wide.head(8)

,feature_name,bucket,control_rate,treatment_rate,lift,lift_se,n_t,n_c
0,f0,1,0.001490,0.001957,0.000467,0.000058,2937322,557576
1,f0,2,0.005651,0.009046,0.003395,0.000121,3012341,482557
2,f0,3,0.000609,0.000803,0.000194,0.000038,2968161,526737
3,f0,4,0.000347,0.000448,0.000101,0.000028,2964831,530067
4,f1,1,0.001532,0.002993,0.001461,0.000055,2689573,805325
5,f1,2,0.001665,0.002572,0.000907,0.000063,2956803,538095
6,f1,3,0.002006,0.002531,0.000525,0.000093,3241674,253224
7,f1,4,0.002848,0.004292,0.001443,0.000084,2994605,500293


In [7]:
# Global conversion ATE from Day 4 — recompute here to keep cells self-contained
global_ate = con.execute("""
    SELECT
        AVG(CASE WHEN treatment = 1 THEN conversion END)
      - AVG(CASE WHEN treatment = 0 THEN conversion END) AS ate
    FROM criteo
""").fetchone()[0]

# Weighted by total bucket size (n_t + n_c)
het_wide["bucket_total"] = het_wide["n_t"] + het_wide["n_c"]

# Weighted mean of bucket lifts, computed per feature, should ≈ global ATE
print(f"Global conversion ATE: {global_ate:.8f}\n")
for feat in features:
    sub = het_wide[het_wide.feature_name == feat]
    weighted_lift = (sub["lift"] * sub["bucket_total"]).sum() / sub["bucket_total"].sum()
    diff_pct = (weighted_lift - global_ate) / global_ate * 100
    print(f"  {feat}: weighted_lift = {weighted_lift:.8f}  (diff vs global: {diff_pct:+.2f}%)")

Global conversion ATE: 0.00115187

  f0: weighted_lift = 0.00103910  (diff vs global: -9.79%)
  f1: weighted_lift = 0.00108393  (diff vs global: -5.90%)
  f2: weighted_lift = 0.00151857  (diff vs global: +31.83%)
  f3: weighted_lift = 0.00118853  (diff vs global: +3.18%)
  f4: weighted_lift = 0.00185737  (diff vs global: +61.25%)
  f5: weighted_lift = 0.00100646  (diff vs global: -12.62%)
  f6: weighted_lift = 0.00110115  (diff vs global: -4.40%)
  f7: weighted_lift = 0.00119268  (diff vs global: +3.54%)
  f8: weighted_lift = 0.00108011  (diff vs global: -6.23%)
  f9: weighted_lift = 0.00134649  (diff vs global: +16.90%)
  f10: weighted_lift = 0.00089383  (diff vs global: -22.40%)
  f11: weighted_lift = 0.00106926  (diff vs global: -7.17%)


In [8]:
FIG_DIR = PROJECT_ROOT / "docs" / "figures"

results_path = FIG_DIR / "heterogeneity_results.csv"
het_wide.drop(columns=["bucket_total"]).to_csv(results_path, index=False)
print(f"Saved {results_path}  ({len(het_wide)} rows)")

Saved /Users/madhumithakatam/Documents/Projects/CriteoUpliftModeling/criteo_uplift_modeling/docs/figures/heterogeneity_results.csv  (48 rows)


In [9]:
# Heterogeneity score = variance of lift across the 4 buckets, per feature.
# Higher variance = more differential treatment response = better signal for uplift.
ranked = (
    het_wide.groupby("feature_name")
    .agg(
        lift_min=("lift", "min"),
        lift_max=("lift", "max"),
        lift_mean=("lift", "mean"),
        heterogeneity_score=("lift", "var"),
    )
    .reset_index()
    .sort_values("heterogeneity_score", ascending=False)
    .reset_index(drop=True)
)
ranked["rank"] = ranked.index + 1
ranked["lift_range"] = ranked["lift_max"] - ranked["lift_min"]
ranked = ranked[["rank", "feature_name", "heterogeneity_score",
                 "lift_min", "lift_max", "lift_range", "lift_mean"]]
ranked

,rank,feature_name,heterogeneity_score,lift_min,lift_max,lift_range,lift_mean
0,1,f4,8.242418e-06,0.000310,0.006162,0.005852,0.001857
1,2,f2,7.797857e-06,0.000030,0.005703,0.005673,0.001519
2,3,f9,5.939862e-06,0.000063,0.005002,0.004938,0.001346
3,4,f3,4.308516e-06,-0.000266,0.004257,0.004523,0.001189
4,5,f8,3.865547e-06,0.000026,0.004026,0.004000,0.001080
5,6,f0,2.490871e-06,0.000101,0.003395,0.003294,0.001039
6,7,f6,2.392761e-06,0.000129,0.003411,0.003282,0.001101
7,8,f7,1.061048e-06,0.000477,0.002722,0.002245,0.001193
8,9,f10,1.055971e-06,0.000314,0.002433,0.002119,0.000894
9,10,f11,5.584945e-07,0.000615,0.002186,0.001572,0.001069


In [10]:
rank_path = FIG_DIR / "feature_rank.csv"
ranked.to_csv(rank_path, index=False)
print(f"Saved {rank_path}")

# Top 5 callout for the writeup
top5 = ranked.head(5)
print("\nTop 5 features by heterogeneity score:")
for _, row in top5.iterrows():
    print(f"  {int(row['rank'])}. {row['feature_name']}: "
          f"score={row['heterogeneity_score']:.2e}, "
          f"lift range [{row['lift_min']:.6f}, {row['lift_max']:.6f}]")

Saved /Users/madhumithakatam/Documents/Projects/CriteoUpliftModeling/criteo_uplift_modeling/docs/figures/feature_rank.csv

Top 5 features by heterogeneity score:
  1. f4: score=8.24e-06, lift range [0.000310, 0.006162]
  2. f2: score=7.80e-06, lift range [0.000030, 0.005703]
  3. f9: score=5.94e-06, lift range [0.000063, 0.005002]
  4. f3: score=4.31e-06, lift range [-0.000266, 0.004257]
  5. f8: score=3.87e-06, lift range [0.000026, 0.004026]


In [11]:
con.close()
print("Connection closed.")

Connection closed.
